In [68]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

df = pd.read_csv(os.path.join(path, "HAM10000_metadata.csv"))

img_dir1 = os.path.join(path, "HAM10000_images_part_1")
img_dir2 = os.path.join(path, "HAM10000_images_part_2")

def get_path(img_id):
    p1 = os.path.join(img_dir1, img_id + ".jpg")
    p2 = os.path.join(img_dir2, img_id + ".jpg")
    return p1 if os.path.exists(p1) else p2

df['path'] = df['image_id'].apply(get_path)

# Binary labels
malignant = ['mel', 'bcc', 'akiec']
df['label'] = df['dx'].apply(lambda x: 1 if x in malignant else 0)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.


In [69]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

In [70]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [71]:
from PIL import Image
from torch.utils.data import Dataset

class SkinDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'path']
        label = self.df.loc[idx, 'label']

        img = Image.open(img_path).convert("RGB")


        if self.transform:
            img = self.transform(img)

        return img, label

In [73]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    SkinDataset(train_df, train_transform),
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SkinDataset(val_df, val_transform),
    batch_size=64,
    shuffle=False
)

In [75]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SkinModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)


        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 1)
)

    def forward(self, x):
        return self.backbone(x)

model = SkinModel().to(device)

In [76]:
def tta_predict(model, img):
    img_flip = torch.flip(img, dims=[3])

    p1 = torch.sigmoid(model(img))
    p2 = torch.sigmoid(model(img_flip))

    return (p1 + p2) / 2

In [77]:
# Freeze most layers
for param in model.backbone.features[:-2].parameters():
    param.requires_grad = False

# Train last layers
for param in model.backbone.features[-2:].parameters():
    param.requires_grad = True

In [78]:
pos_weight = torch.tensor([1.3]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [79]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=5
)

In [80]:
imgs, labels = next(iter(train_loader))
print(type(imgs))

<class 'torch.Tensor'>


In [81]:
scaler = torch.amp.GradScaler("cuda")

for epoch in range(8):
    model.train()

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(imgs).squeeze()
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done


In [82]:
import numpy as np

model.eval()
probs = []
y_val = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)

        outputs = torch.sigmoid(model(imgs)).cpu().numpy()

        probs.extend(outputs)
        y_val.extend(labels.numpy())

probs = np.array(probs).flatten()
y_val = np.array(y_val)

In [83]:
from sklearn.metrics import accuracy_score
import numpy as np

best_acc = 0
best_t = 0

for t in np.arange(0.3, 0.7, 0.01):
    preds = (probs > t).astype(int)
    acc = accuracy_score(y_val, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best threshold:", best_t)

Best threshold: 0.5200000000000002


In [84]:
class SmoothBCE(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return nn.BCEWithLogitsLoss()(logits, targets)

criterion = SmoothBCE()

In [85]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_pred = (probs > best_t).astype(int)

print("Accuracy :", accuracy_score(y_val, y_pred))
print("Precision:", precision_score(y_val, y_pred))
print("Recall   :", recall_score(y_val, y_pred))

Accuracy : 0.8756864702945582
Precision: 0.7028571428571428
Recall   : 0.629156010230179
